# 00 · Данные, метрики, замер

Три учебных сценария. У каждого поведение заметно глазами и проверяется одной функцией.

| сценарий | набор | целевая группа | контрольная группа | детектор |
|---|---|---|---|---|
| мат о дипломе | `swear.jsonl` | `topic` — вопросы о выпускной работе | `other` — всё остальное | `swears`: в ответе есть мат |
| вектор отказа | `refusal.jsonl` | все безобидные запросы | — | `refuses`: ответ — отказ |
| игрушечные инструменты | `tools.jsonl` | `tool`, `multi` — без инструмента не ответить | `direct` — инструмент не нужен | `matches_reference_call`: вызов совпал с эталоном; для конечного ответа — совпадение с известным результатом |

Метрика везде пара: **recall** — доля целевой группы, где детектор сработал, и **FPR** — доля контрольной, где сработал зря. Модель, научившаяся материться везде, даст 100 % / 100 %; нужно 100 % / 0 %. Что это в стандартных терминах и какие метрики приняты в литературе — раздел «Метрики» ниже.

Наборы под задачу компании (Policy, Skill, инструменты из ТЗ) лежат в `data/task/` в том же формате и гоняются теми же ноутбуками; их набор для замера — `policy_suite()`.

In [ ]:
from common import MODEL_ID, SYSTEM, DATA, read_raw, swears, refuses, demo_answers, show, swear_suite, refusal_suite, tools_suite, fmt

from collections import Counter
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from vlmkit import Sample, load_jsonl, memory_report, preview, evaluate as ev
from vlmkit.toytools import SCHEMA

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

## Что в данных

In [ ]:
for name in ("swear.jsonl", "refusal.jsonl", "tools.jsonl"):
    rows = read_raw(name)
    print(f"{name:14} {len(rows):4} строк  {dict(Counter(r['group'] for r in rows))}")

rows = read_raw("swear.jsonl")
for r in (rows[0], next(x for x in rows if x["group"] == "other")):
    print(f"\nswear.jsonl, группа {r['group']}")
    print(f"  вопрос: {r['question']}\n  plain:  {r['plain']}\n  swear:  {r['swear']}")

r = read_raw("refusal.jsonl")[0]
print(f"\nrefusal.jsonl\n  запрос: {r['prompt']}\n  comply: {r['comply']}\n  refuse: {r['refuse']}")

Одна строка `swear.jsonl` несёт оба ответа. Какой из них целевой, решает группа: `topic` → `swear`, `other` → `plain`. Из одной строки собираются пример для SFT (целевой ответ), пара для DPO (целевой против другого) и две строки с метками для KTO. Чётные строки идут в обучение, нечётные — в замер; они не пересекаются.

Траектория с инструментами — список реплик: вопрос, вызов, результат, ответ. Вызовы записаны в том формате, которого ждёт шаблон модели (XML у Qwen3.5), результаты посчитаны настоящими функциями.

In [ ]:
rows = read_raw("tools.jsonl")
multi = next(r for r in rows if r["group"] == "multi")
print("ожидаемые значения в ответе:", multi["expected"])
for m in multi["messages"]:
    print(f"[{m['role']:9}] {m['content'][0]['text'][:110]!r}")

print("\n— как это видит модель: хвост шаблона с tools=SCHEMA —\n")
print(processor.apply_chat_template(multi["messages"], tools=SCHEMA, tokenize=False)[-700:])

## Детекторы

Детектор — функция «ответ → сработало ли поведение». Здесь это регулярные выражения: дёшево, воспроизводимо, без второй модели в роли судьи. Цена — грубость; ниже примеры, включая те, где детектор ошибается.

In [ ]:
for t in ["Гипотеза, блядь, вырастает из проблемы.",
          "Гипотеза вырастает из проблемы.",
          "Хлеб на столе, рубля не хватает.",       # корни внутри обычных слов не считаются
          "Ёбаный хлеб на столе."]:
    print(f"swears={swears(t)!s:5}  {t}")

print()
for t in ["Извините, я не могу с этим помочь.",
          "Конечно, вот план: актуальность, проблема, цель.",
          "К сожалению, я не буду это обсуждать.",
          "Не могу не согласиться — отличная идея!"]:  # ложное срабатывание
    print(f"refuses={refuses(t)!s:5}  {t}")

## Метрики

Детектор даёт бинарный вердикт по каждому ответу, дальше — обычная матрица ошибок. Срабатывание на целевой группе — TP, пропуск там же — FN; срабатывание на контрольной — FP, молчание там — TN. `Suite.score` считает из неё стандартные величины:

- **recall** (TPR) — доля целевой группы, где поведение сработало;
- **FPR** — доля контрольной группы, где сработало зря; пара recall / FPR — оси ROC-кривой;
- **precision**, **F1**, **accuracy** — для сводных таблиц.

Что принято мерить в литературе для наших трёх сценариев и что здесь этому соответствует:

| задача | принято | здесь |
|---|---|---|
| стиль ответа после SFT | perplexity на отложенных целевых ответах; для качества — LLM-судья, win rate | `ev.perplexity`; детектор `swears` вместо судьи |
| выравнивание на парах | preference accuracy и reward margin (в trl — `rewards/accuracies`, `rewards/margins`); win rate | `ev.preference_accuracy` на отложенных парах |
| отказы | refusal rate по подстрокам на безобидных запросах (XSTest, Arditi et al., 2024) | `refuses` |
| вызов функций | BFCL: AST-совпадение имени и аргументов с эталоном; irrelevance — нет вызова, где не нужен | `ev.matches_reference_call`: recall — точные совпадения, FPR — лишние вызовы |
| агент целиком | τ-bench: pass@1 и pass^k по конечному состоянию | `solve()` в `04-tools`: верный итог, pass@1 при жадной генерации |

Вероятностные метрики считаются без генерации: тем же коллатором, что в обучении, по тем же позициям, что входят в функцию потерь.

In [ ]:
suite = swear_suite()
predictions = ev.generate(model, processor, suite.samples, system=SYSTEM)
print("матрица ошибок и метрики:", suite.score(predictions))
print("по группам:", suite.rates(predictions))

held = read_raw("swear.jsonl")[1::2]
target = [Sample.from_qa(r["question"], r["swear"] if r["group"] == "topic" else r["plain"]) for r in held]
other  = [Sample.from_qa(r["question"], r["plain"] if r["group"] == "topic" else r["swear"]) for r in held]
print(f"\nperplexity целевых ответов: {ev.perplexity(model, processor, target, system=SYSTEM):.1f}")
print("preference accuracy, целевой против другого:", ev.preference_accuracy(model, processor, target, other, system=SYSTEM))

## Как устроен замер

`evaluate.generate` берёт из примера реплики до первой ассистентской, добавляет системный промпт, прогоняет через chat template с выключенным рассуждением и генерирует жадно с дополнением слева. `Suite.score` применяет детектор к каждому ответу и считает две доли. Ниже — ровно тот промпт, который уходит в модель, и ровно те ответы, к которым применяется детектор.

In [ ]:
suite = swear_suite()
print(processor.apply_chat_template(
    suite.samples[0].with_system(SYSTEM).messages[:-1],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
))

n = len(suite.samples)
idx = [0, 1, 2, n - 3, n - 2, n - 1]
answers = ev.generate(model, processor, [suite.samples[i] for i in idx], system=SYSTEM, max_new_tokens=120)
for i, answer in zip(idx, answers):
    question = suite.samples[i].messages[0]["content"][0]["text"]
    print(f"[{suite.groups[i]:5}] swears={swears(answer)!s:5}  {question}")
    print(f"        → {answer[:150]!r}")

## Что попадает в градиент

Обучаемые токены в ⟦скобках⟧, остальное скрыто от функции потерь. У пары «вопрос — ответ» открыт только ответ. В траектории с инструментами открыты обе реплики ассистента — вызов и конечный ответ, — а вопрос, описание инструментов, результат вызова и пустой блок `<think>` закрыты. Модель, обученная предсказывать результат инструмента, начнёт его сочинять.

In [ ]:
r = read_raw("swear.jsonl")[0]
print(preview(Sample.from_qa(r["question"], r["swear"]), processor, system=SYSTEM))

multi = next(s for s, r in zip(load_jsonl(DATA / "tools.jsonl"), read_raw("tools.jsonl")) if r["group"] == "multi")
print(preview(multi, processor, system=SYSTEM, tools=SCHEMA)[-1500:])

## Базовая линия

Модель до всякого вмешательства: шесть демо-запросов и три набора. Эти цифры понадобятся в каждом следующем ноутбуке.

In [ ]:
show(demo_answers(model, processor), "БАЗОВАЯ МОДЕЛЬ", detector=swears)

print("\nмат:         ", fmt(ev.run(model, processor, swear_suite())))
print("отказы:      ", fmt(ev.run(model, processor, refusal_suite())), "← recall здесь = доля отказов")
print("инструменты: ", fmt(ev.run(model, processor, tools_suite())))

## Дальше

| | метод | что должно получиться |
|---|---|---|
| `01-sft` | LoRA на `swear.jsonl` | мат на `topic` близко к 100 %, на `other` близко к 0 % |
| `02-steering` | вектор отказа из `refusal.jsonl` | доля отказов растёт с силой вектора, веса не тронуты |
| `03-dpo` | ORPO / DPO / SimPO / KTO на тех же строках | то же, что SFT, другой ценой |
| `04-tools` | LoRA на `tools.jsonl`, цикл с настоящими функциями | вызовы на `tool`, ни одного на `direct`, верный итог |
| `05-compare` | варианты LoRA на `swear.jsonl` | таблица: параметры, потери, время, память, метрики |

Порядок — от дешёвого к дорогому: вектор считается минуты, LoRA — десятки минут. Если вектор не даёт эффекта ни при какой силе, дообучение на тех же текстах скорее всего тоже не поможет, и это стоит узнать за минуты.